In [26]:
import os
import glob
import tiktoken
import numpy as np
import pandas as pd
import gradio as gr
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import TextLoader, PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_ollama import ChatOllama


In [27]:
MODEL = "gpt-5.4"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv("OPENAI_API_KEY")

if openai_api_key is None:
    raise ValueError("OPENAI_API_KEY is not set")

In [28]:
folders = glob.glob("knowledge-base/*")
documents = []

for folder in folders:
    folder_name = os.path.basename(folder).lower()

    if folder_name == "resume":
        pdf_files = glob.glob(f"{folder}/*.pdf")
        for pdf in pdf_files:
            loader = PyPDFLoader(file_path=pdf)
            folder_docs = loader.load()

            for doc in folder_docs:
                doc.metadata["doc_type"] = "resume"
                doc.metadata["source"] = pdf
                documents.append(doc)

    else:
        loader = DirectoryLoader(
            folder,
            glob="*.md",
            loader_cls=TextLoader,
            loader_kwargs={"encoding": "utf-8"}
        )

        folder_docs = loader.load()

        for doc in folder_docs:
            doc.metadata["doc_type"] = folder_name
            doc.metadata["source"] = doc.metadata.get("source", "")
            documents.append(doc)

print(f"Loaded {len(documents)} documents")

Loaded 15 documents


In [29]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 53 chunks
First chunk:

page_content='# Simon-Game
Link to the game: https://niloysaha84.github.io/Simon-Game/' metadata={'source': 'knowledge-base/Projects/NiloySaha84_Simon-Game.md', 'doc_type': 'projects'}


In [30]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Vectorstore created with 53 documents


In [31]:
collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 53 vectors with 384 dimensions in the vector store


In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 7})
# llm = ChatOpenAI(model=MODEL, temperature=0)
llm = ChatOllama(model="qwen2.5:3b")

In [42]:
SYSTEM_PROMPT_TEMPLATE = """
You are an AI assistant representing Niloy Saha on his personal portfolio website.

Your job is to answer questions about Niloy using ONLY the provided context (retrieved documents such as resume, project READMEs, and other indexed data). Do NOT make up information. If the answer is not in the context, say you don’t have enough information.

Always speak in a professional, concise, and friendly tone.

IMPORTANT RULES:

1. IDENTITY:
- This website belongs to Niloy Saha.
- If users refer to "you", interpret it as Niloy Saha (not the AI).

2. CONTEXT USAGE:
- Base your answers strictly on the retrieved context.
- Do not hallucinate or assume missing details.
- If unsure, say: "I don't have enough information about that."

3. PROJECT QUESTIONS:
- If the user asks about projects, list relevant projects from context.
- After listing, ALWAYS include:
  "You can explore more projects on GitHub: {GITHUB_LINK} or visit the projects section: {PROJECTS_PAGE_LINK}."

4. EXPERIENCE / EDUCATION QUESTIONS:
- If the user asks about studies, education, or work experience:
  - Summarize clearly using the context.
  - Then ALWAYS include:
  "For more details, you can check his GitHub: {GITHUB_LINK} or visit his portfolio: {EDUCATION_LINK}."

5. SKILLS QUESTIONS:
- If the user asks about skills, list relevant skills from context.
- Then ALWAYS include:
  "For more details, you can check his GitHub: {GITHUB_LINK} or visit his portfolio: {SKILLS_LINK}."

6. STYLE:
- Keep answers short and structured.
- Use bullet points when listing items.
- Avoid long paragraphs.

7. PERSONAL QUESTIONS:
- Answer only if information is available in the context.
- Do not fabricate personality traits or personal life details.

8. FALLBACK:
- If no relevant information is found:
  "I couldn't find that information in Niloy's portfolio data."

9. NO META TALK:
- Do not mention "context", "RAG", or "documents".
- Do not say "based on the provided data".

EXAMPLE:

Q: What projects has he done?
A:
Some of Niloy Saha's projects include:
- Dragon Slayer Game
- RAG Chatbot for Portfolio Website
- AI Web Page Summarizer Chrome Extension
- AI Voice Study Companion
- [Other project from context]

You can explore more projects on GitHub: {GITHUB_LINK} or visit the projects section: {PROJECTS_PAGE_LINK}.

Context: {context}
Useful Links:
- GitHub: {GITHUB_LINK}
- Projects Page: {PROJECTS_PAGE_LINK}
- Education Page: {EDUCATION_LINK}
- Experience Page: {EXPERIENCE_LINK}
- Skills Page: {SKILLS_LINK}
- Contact Page: {CONTACT_LINK}

"""

In [43]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context,  GITHUB_LINK= "https://github.com/niloy-saha",
    ABOUT_LINK= "https://niloy-saha84-github-io.vercel.app/#about",
    EDUCATION_LINK= "https://niloy-saha84-github-io.vercel.app/#education",
    EXPERIENCE_LINK= "https://niloy-saha84-github-io.vercel.app/#experience",
    PROJECTS_PAGE_LINK= "https://niloy-saha84-github-io.vercel.app/#projects",
    SKILLS_LINK= "https://niloy-saha84-github-io.vercel.app/#skills",
    CONTACT_LINK= "https://niloy-saha84-github-io.vercel.app/#contact")
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [44]:
theme = gr.themes.Origin(
    # primary_hue="sky",
    # secondary_hue="gray",
)
gr.ChatInterface(answer_question, title="Chatbot", description="Hey! Want a quick tour of my projects or skills? Just ask.", theme=theme).launch(share=True)

/Users/niloysaha/.local/share/uv/python/cpython-3.12.12-macos-aarch64-none/lib/python3.12/site-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://61cbf881c3a9817e40.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
